In [1]:
import os
from dotenv import load_dotenv
from src.ingest import ingest_documents
from src.retriever import get_collection, retrieve
from src.rag_pipeline import rag_answer
from src.structured import generate_structured_output
import pandas as pd

load_dotenv()
print("Ready.")

Ready.


In [2]:
import os
from src.ingest import ingest_documents

PDF_DIR = 'data/pdfs/'
pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files: {pdf_files}")
print("\nIngesting PDFs into ChromaDB...")

collection = ingest_documents(PDF_DIR)
print(f"\nDone. Collection has {collection.count()} chunks stored.")
print("You can now run the remaining cells.")

Found 3 PDF files: ['CE308_ProjectReport.pdf', 'ProjectReport_2023037_2023694.pdf', 'Raid Storage Study Guide.pdf']

Ingesting PDFs into ChromaDB...
Loaded: CE308_ProjectReport.pdf - 15896 characters
Loaded: ProjectReport_2023037_2023694.pdf - 19542 characters
Loaded: Raid Storage Study Guide.pdf - 8513 characters
CE308_ProjectReport.pdf: 36 chunks
ProjectReport_2023037_2023694.pdf: 44 chunks
Raid Storage Study Guide.pdf: 19 chunks
Stored 99 chunks in ChromaDB

Done. Collection has 99 chunks stored.
You can now run the remaining cells.


In [3]:
from src.retriever import get_collection, retrieve

collection = get_collection()
query = "What is the main topic of the document?"
chunks, sources = retrieve(query, collection)

print(f"Query: {query}\n")
for i, (chunk, source) in enumerate(zip(chunks, sources)):
    print(f"--- Chunk {i+1} from {source} ---")
    print(chunk[:300])
    print()

Query: What is the main topic of the document?

--- Chunk 1 from ProjectReport_2023037_2023694.pdf ---
pirational References 
References were drawn from data journalism, print, UI design, and books to ensure a 
diverse design vocabulary: 
Source Medium Design Element Borrowed 
NYT ‘The Air We 
Breathe’ (2023) 
Web / Data 
Journalism 
Dark background with AQI color scale; 
scrollytelling pacing and an

--- Chunk 2 from ProjectReport_2023037_2023694.pdf ---
el 
• Progress bar: Fixed at top — shows scroll position through the full story (0–
100%) 
• 8 nav dots: Fixed on right side — click any dot to jump directly to that scene 
• Scenes 1–3 and 8: Author-driven — content appears and animates on scroll 
• Scenes 4–6: Hybrid — charts animate on scroll-ent

--- Chunk 3 from ProjectReport_2023037_2023694.pdf ---
ts 
The Pudding — 
Visual Essays Web Full-viewport sections; smooth transitions; 
reader engagement hooks 
Bloomberg CityLab 
Smog Coverage 
Web / 
Newsprint 
Annotated line charts; n

In [4]:
from src.constants import AGENT_COLOR, USER_COLOR, RESET, BOLD, INFO_COLOR

questions = [
    "What is the main topic of the document?",
    "Summarize the key points.",
    "What conclusions are drawn?"
]

for question in questions:
    print(f"{USER_COLOR}{BOLD}Question:{RESET} {question}")
    answer, sources, chunks = rag_answer(question, collection)
    print(f"{AGENT_COLOR}{BOLD}Answer:{RESET} {answer}")
    print(f"{INFO_COLOR}Sources: {set(sources)}{RESET}")
    print()

Question: What is the main topic of the document?
Answer: Based on the context provided, the document appears to be a **design specification and inspirational reference guide for a data visualization/scrollytelling project focused on air quality**. It lists references from sources like the NYT, IQAir, and Bloomberg CityLab (all related to air quality/smog) and details UI/UX components such as a progress bar, navigation dots, scene behaviors, and transitions.
Sources: {'ProjectReport_2023037_2023694.pdf'}

Question: Summarize the key points.
Answer: Based on the provided context, here are the key points:

**Visualization Scenes:**
- **Scene 6 (24-Hour Daily Cycle):** Radial clock chart with 24 segments showing average AQI by hour; bars animate outward sequentially to complete the clock face.
- **Scene 7 (Free Explorer):** Interactive dashboard with Year dropdown (All/2021/2022/2023/2024) and Metric dropdown (AQI/PM2.5); real-time chart updates, hover tooltips, and a reset button.

**Vis

In [5]:
from src.constants import AGENT_COLOR, USER_COLOR, ERROR_COLOR, INFO_COLOR, RESET, BOLD
movies = [
    "The Dark Knight - a superhero film about Batman fighting the Joker in Gotham City",
    "Titanic - a romance film about two people falling in love on a doomed ship",
    "Interstellar - a sci-fi film about astronauts travelling through a wormhole to save humanity"
]

results = []
for movie in movies:
    validated, raw, error = generate_structured_output(movie)
    if validated:
        print(f"{USER_COLOR}{BOLD}Movie:{RESET} {validated.title}")
        print(f"{AGENT_COLOR}{BOLD}Genre:{RESET} {validated.genre}")
        print(f"{AGENT_COLOR}{BOLD}Rating:{RESET} {validated.rating}")
        print(f"{AGENT_COLOR}{BOLD}Recommended:{RESET} {validated.recommended}")
        results.append(validated.model_dump())
    else:
        print(f"{ERROR_COLOR}Validation failed: {error}{RESET}")
        print(f"Raw output: {raw}")
    print()

Validation failed: unhashable type: 'dict'
Raw output: 

Validation failed: unhashable type: 'dict'
Raw output: 

Validation failed: unhashable type: 'dict'
Raw output: 



In [6]:
if results:
    df = pd.DataFrame(results)
    df

In [7]:
print(f"""{USER_COLOR}{BOLD}Hallucination Analysis{RESET}

1. Retrieved chunks are irrelevant - model makes up an answer
   Detection: check if answer contains words from retrieved chunks

2. Model ignores context and uses its own knowledge
   Detection: ask about something NOT in the documents

3. Structured output has wrong values even if schema is valid
   Detection: rating of 15.0 passes schema but is out of range
   Fix: add validator in Pydantic using field_validator
""")

Hallucination Analysis

1. Retrieved chunks are irrelevant - model makes up an answer
   Detection: check if answer contains words from retrieved chunks

2. Model ignores context and uses its own knowledge
   Detection: ask about something NOT in the documents

3. Structured output has wrong values even if schema is valid
   Detection: rating of 15.0 passes schema but is out of range
   Fix: add validator in Pydantic using field_validator



## Week 3 Findings

### RAG Pipeline
Built a RAG system that loads PDFs, chunks them into 500 character
overlapping segments, stores them in ChromaDB, and retrieves the
most relevant chunks for any query.

### Chunking Strategy
Fixed size chunks of 500 characters with 50 character overlap.
Overlap ensures sentences split across boundaries are still captured.

### Structured Outputs
LLM returns JSON validated against a Pydantic schema.
Missing fields or wrong types raise ValidationError immediately.

### Hallucination Detection
Added explicit prompt instruction to say I do not know if
the answer is not in the context. Schema validation catches
type errors but not value errors.